# **Object Detection: YOLOv11 on COCO Subset**

Notebook này train trên **subset của validation set** được chia thành 80/20 (training/validation).

Goals:
- Train và compare **YOLO11n** và **YOLO11s** trên cùng subset.
- Chia validation set thành 80% training, 20% validation.
- Lưu kết quả ở `assignment_2/runs/assignment2_yolov11_coco_subset`.
- Resume sau Colab quota interruption bằng `last.pt`.



## **1. Chuẩn bị môi trường**
Cài đặt các thư viện cần thiết. Cell này cần `sklearn` để split dataset.


In [5]:
%pip install -q ultralytics==8.3.0 pycocotools pandas seaborn pyyaml scikit-learn



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [6]:
import json
import random
from pathlib import Path
import os

os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"
os.environ["WANDB_SILENT"] = "true"

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
from IPython.display import display
from ultralytics import YOLO
from ultralytics.data.utils import check_det_dataset

sns.set_theme(style='whitegrid')
random.seed(42)
np.random.seed(42)

## **Local output setup**
- Không dùng Google Drive.
- Toàn bộ kết quả lưu ở `assignment_2/runs`.

In [7]:
def resolve_runs_root():
    cwd = Path.cwd()
    if cwd.name == 'assignment_2':
        return cwd / 'runs'
    if (cwd / 'assignment_2').exists():
        return cwd / 'assignment_2' / 'runs'
    return cwd / 'assignment_2' / 'runs'


RUNS_ROOT = resolve_runs_root()
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

print(f'RUNS_ROOT={RUNS_ROOT.resolve()}')


RUNS_ROOT=/Users/ngonhattoan/IdeaProjects/CO5085_DeepLearning_CV/assignment_2/runs


## **2. Thiết lập bài toán và dữ liệu**
- Notebook này train trên **COCO subset** được tạo từ validation set.
- Subset được chia **80% training / 20% validation**.
- Kết quả sẽ lưu ở `assignment_2/runs/assignment2_yolov11_coco_subset`.


In [8]:
USE_COCO128_DEBUG = False  # False: full COCO subset, True: quick debug with coco128
DATA_YAML = 'coco_subset.yaml'  # Using subset from validation set (80/20 split)

# Persist all runs under assignment_2/runs/assignment2_yolov11_coco_subset
PROJECT_DIR = RUNS_ROOT / 'assignment2_yolov11_coco_subset'
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

data_info = check_det_dataset(DATA_YAML)
print(f'Using dataset config: {DATA_YAML}')
print(f'Project dir: {PROJECT_DIR}')
data_info


Using dataset config: coco_subset.yaml
Project dir: /Users/ngonhattoan/IdeaProjects/CO5085_DeepLearning_CV/assignment_2/runs/assignment2_yolov11_coco_subset


{'path': PosixPath('/Users/ngonhattoan/IdeaProjects/datasets/coco/coco_subset'),
 'train': '/Users/ngonhattoan/IdeaProjects/datasets/coco/coco_subset/train/images',
 'val': '/Users/ngonhattoan/IdeaProjects/datasets/coco/coco_subset/val/images',
 'nc': 80,
 'names': {0: 'person',
  1: 'bicycle',
  2: 'car',
  3: 'motorcycle',
  4: 'airplane',
  5: 'bus',
  6: 'train',
  7: 'truck',
  8: 'boat',
  9: 'traffic light',
  10: 'fire hydrant',
  11: 'stop sign',
  12: 'parking meter',
  13: 'bench',
  14: 'bird',
  15: 'cat',
  16: 'dog',
  17: 'horse',
  18: 'sheep',
  19: 'cow',
  20: 'elephant',
  21: 'bear',
  22: 'zebra',
  23: 'giraffe',
  24: 'backpack',
  25: 'umbrella',
  26: 'handbag',
  27: 'tie',
  28: 'suitcase',
  29: 'frisbee',
  30: 'skis',
  31: 'snowboard',
  32: 'sports ball',
  33: 'kite',
  34: 'baseball bat',
  35: 'baseball glove',
  36: 'skateboard',
  37: 'surfboard',
  38: 'tennis racket',
  39: 'bottle',
  40: 'wine glass',
  41: 'cup',
  42: 'fork',
  43: 'knife',


In [9]:
import zipfile

# Chỉ extract labels nếu sử dụng coco.yaml hoặc coco128.yaml
if DATA_YAML in ['coco.yaml', 'coco128.yaml']:
    labels_zip_path = Path(data_info['path']).parent / 'coco2017labels-segments.zip'
    labels_extract_dir = Path(data_info['path'])

    if labels_zip_path.exists():
        print(f'Extracting labels from {labels_zip_path} to {labels_extract_dir}...')
        with zipfile.ZipFile(labels_zip_path, 'r') as zip_ref:
            zip_ref.extractall(labels_extract_dir)
        print('Labels extracted successfully.')
    else:
        print(f'Labels zip file not found at {labels_zip_path}. Please ensure it was downloaded.')
else:
    print(f'Using {DATA_YAML} - labels đã được tạo từ validation set, skip extraction.')


Using coco_subset.yaml - labels đã được tạo từ validation set, skip extraction.


## **Tạo subset từ validation set (80/20 split)**
Sử dụng validation set và chia thành 80% training và 20% validation cho subset này.

In [10]:
import shutil
from sklearn.model_selection import train_test_split

# Define helper functions
def resolve_path(base_path, value):
    p = Path(value)
    return p if p.is_absolute() else (Path(base_path) / p)

def get_images_from_list_file(list_file_path, base_data_path):
    images = []
    if list_file_path.is_file() and list_file_path.suffix.lower() == '.txt':
        with open(list_file_path, 'r', encoding='utf-8') as f:
            for line in f:
                s = line.strip()
                if not s:
                    continue
                p = Path(s)
                if not p.is_absolute():
                    p = Path(base_data_path) / p
                if p.exists() and p.is_file():
                    images.append(p)
    elif list_file_path.is_dir():
        images = sorted(
            list(list_file_path.glob('*.jpg')) +
            list(list_file_path.glob('*.jpeg')) +
            list(list_file_path.glob('*.png'))
        )
    return sorted(images)

# Get validation images list
val_list_file = resolve_path(data_info.get('path', '.'), data_info['val'])
val_images_list = get_images_from_list_file(val_list_file, data_info.get('path', '.'))
print(f'Total validation images: {len(val_images_list)}')

# Split 80/20
train_subset, val_subset = train_test_split(
    val_images_list, 
    test_size=0.2, 
    random_state=42
)

print(f'Train subset (80%): {len(train_subset)} images')
print(f'Val subset (20%): {len(val_subset)} images')

# Create folder structure for subset
subset_root = Path(data_info['path']) / 'coco_subset'
subset_root.mkdir(exist_ok=True)

train_subset_dir = subset_root / 'train' / 'images'
val_subset_dir = subset_root / 'val' / 'images'
train_labels_dir = subset_root / 'train' / 'labels'
val_labels_dir = subset_root / 'val' / 'labels'

for d in [train_subset_dir, val_subset_dir, train_labels_dir, val_labels_dir]:
    d.mkdir(parents=True, exist_ok=True)

# Detect validation split folder name
val_split_folder = val_images_list[0].parent.name if val_images_list else 'val2017'
print(f'\nDetected validation split folder: {val_split_folder}')

# Copy train subset images and labels
print('Copying train subset images and labels...')
for img_path in train_subset:
    img_dest = train_subset_dir / img_path.name
    shutil.copy2(img_path, img_dest)
    
    # Copy corresponding label
    label_src = img_path.parent.parent / 'labels' / val_split_folder / img_path.stem + '.txt'
    if label_src.exists():
        label_dest = train_labels_dir / img_path.stem + '.txt'
        shutil.copy2(label_src, label_dest)

print(f'✓ Copied {len(train_subset)} train images')

# Copy val subset images and labels
print('Copying val subset images and labels...')
for img_path in val_subset:
    img_dest = val_subset_dir / img_path.name
    shutil.copy2(img_path, img_dest)
    
    # Copy corresponding label
    label_src = img_path.parent.parent / 'labels' / val_split_folder / img_path.stem + '.txt'
    if label_src.exists():
        label_dest = val_labels_dir / img_path.stem + '.txt'
        shutil.copy2(label_src, label_dest)

print(f'✓ Copied {len(val_subset)} val images')

print(f'\n✓ Subset created at: {subset_root.resolve()}')


Total validation images: 1000
Train subset (80%): 800 images
Val subset (20%): 200 images

Detected validation split folder: images
Copying train subset images and labels...


TypeError: unsupported operand type(s) for +: 'PosixPath' and 'str'

In [ ]:
# Tạo YAML config cho subset
coco_subset_yaml_content = f"""
# COCO Subset Dataset (from val set, 80/20 split)
path: {subset_root.resolve()}
train: train/images
val: val/images

nc: {len(data_info['names'])}
names: {data_info['names']}
"""

subset_yaml_path = Path.cwd() / 'coco_subset.yaml'
with open(subset_yaml_path, 'w', encoding='utf-8') as f:
    f.write(coco_subset_yaml_content)

print(f'✓ Created YAML config: {subset_yaml_path.resolve()}')
print(f'\n{coco_subset_yaml_content}')


## **3. Phân tích tập dữ liệu (EDA)**
Phần này thống kê quy mô dữ liệu và phân bố lớp để kiểm tra mức độ đa dạng cũng như tính mất cân bằng trong tập train.

In [ ]:
def resolve_path(base_path, value):
    p = Path(value)
    return p if p.is_absolute() else (Path(base_path) / p)


train_list_file = resolve_path(data_info.get('path', '.'), data_info['train'])
val_list_file = resolve_path(data_info.get('path', '.'), data_info['val'])


def get_images_from_list_file(list_file_path, base_data_path):
    images = []
    if list_file_path.is_file() and list_file_path.suffix.lower() == '.txt':
        with open(list_file_path, 'r', encoding='utf-8') as f:
            for line in f:
                s = line.strip()
                if not s:
                    continue
                p = Path(s)
                if not p.is_absolute():
                    p = Path(base_data_path) / p
                if p.exists() and p.is_file():
                    images.append(p)
    elif list_file_path.is_dir():
        images = sorted(
            list(list_file_path.glob('*.jpg')) +
            list(list_file_path.glob('*.jpeg')) +
            list(list_file_path.glob('*.png'))
        )
    return sorted(images)


train_images = get_images_from_list_file(train_list_file, data_info.get('path', '.'))
val_images = get_images_from_list_file(val_list_file, data_info.get('path', '.'))

base_data_path = Path(data_info.get('path', '.'))

# Determine label paths based on subset structure (train/labels and val/labels)
label_train_path = base_data_path / 'train' / 'labels'
label_val_path = base_data_path / 'val' / 'labels'

print(f'Train images list file: {train_list_file}')
print(f'Train images: {len(train_images)}')
print(f'Val images list file:   {val_list_file}')
print(f'Val images:   {len(val_images)}')
print(f'Classes:      {len(data_info["names"])}')
print(f'Label train path: {label_train_path}')
print(f'Label val path: {label_val_path}')



In [ ]:
def count_class_distribution(label_dir, class_names):
    counts = {i: 0 for i in range(len(class_names))}
    for lb in sorted(label_dir.glob('*.txt')):
        with open(lb, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split()
                if not parts:
                    continue
                cls_idx = int(float(parts[0]))
                if cls_idx in counts:
                    counts[cls_idx] += 1

    df = pd.DataFrame({
        'class_id': list(counts.keys()),
        'class_name': [class_names[i] for i in counts.keys()],
        'instances': list(counts.values())
    }).sort_values('instances', ascending=False)
    return df


class_names = data_info['names']
dist_df = count_class_distribution(label_train_path, class_names)
display(dist_df.head(15))

plt.figure(figsize=(12, 5))
sns.barplot(data=dist_df.head(15), x='instances', y='class_name', palette='viridis')
plt.title('Top 15 class xuất hiện nhiều nhất (train set)')
plt.xlabel('Số lượng instance')
plt.ylabel('Class')
plt.tight_layout()
plt.show()

Từ biểu đồ phân bố lớp, có thể nhận thấy tần suất xuất hiện giữa các lớp không đồng đều. Đây là đặc điểm phổ biến của COCO và có thể ảnh hưởng trực tiếp đến chất lượng phát hiện ở các lớp hiếm.

Trong quá trình phân tích kết quả, cần theo dõi thêm lỗi false negative/false positive theo từng nhóm lớp để có đánh giá đầy đủ hơn.

## **4. Cấu hình huấn luyện mô hình**
Thiết lập cấu hình chung cho các thí nghiệm nhằm đảm bảo tính công bằng khi so sánh YOLO11n và YOLO11s.

In [ ]:
TRAIN_ARGS = {
    'data': DATA_YAML,
    'imgsz': 640,
    'epochs': 5,
    'batch': 16,
    'workers': 4,
    'project': str(PROJECT_DIR),
    'pretrained': True,
    'optimizer': 'auto',
    'seed': 42,
    'patience': 20,
    'cos_lr': True,
    'close_mosaic': 10,
    'save_period': 1,
    'exist_ok': True
}
TRAIN_ARGS

## **5. Huấn luyện và so sánh hai biến thể YOLOv11**
Phần này đáp ứng yêu cầu cốt lõi của đề bài: **so sánh ít nhất hai phương pháp/biến thể** trên cùng một pipeline dữ liệu và cùng cấu hình đánh giá.

In [ ]:
experiments = {
    'yolo11n': 'yolo11n.pt',
    'yolo11s': 'yolo11s.pt',
}

train_runs = globals().get('train_runs', {})


def get_completed_epochs(exp_name):
    run_dir = PROJECT_DIR / f'{exp_name}_coco'
    results_csv = run_dir / 'results.csv'
    if not results_csv.exists():
        return 0

    try:
        df = pd.read_csv(results_csv)
        if df.empty:
            return 0
        if 'epoch' in df.columns:
            return int(df['epoch'].dropna().max()) + 1
        return int(len(df))
    except Exception as e:
        print(f'[WARN] Không đọc được {results_csv}: {e}')
        return 0


def get_train_status(exp_name):
    target_epochs = int(TRAIN_ARGS.get('epochs', 0))
    completed_epochs = get_completed_epochs(exp_name)
    done = target_epochs > 0 and completed_epochs >= target_epochs
    return done, completed_epochs, target_epochs


def train_or_resume(exp_name, ckpt_name):
    run_name = f'{exp_name}_coco'
    run_dir = PROJECT_DIR / run_name
    last_ckpt = run_dir / 'weights' / 'last.pt'
    best_ckpt = run_dir / 'weights' / 'best.pt'

    if last_ckpt.exists():
        print(f'[RESUME] {exp_name}: {last_ckpt}')
        model = YOLO(str(last_ckpt))
        return model.train(resume=True)

    print(f'[NEW RUN] {exp_name}: {ckpt_name}')
    model = YOLO(str(best_ckpt)) if best_ckpt.exists() else YOLO(ckpt_name)
    run_args = TRAIN_ARGS.copy()
    run_args['name'] = run_name
    return model.train(**run_args)


train_done_flags = globals().get('train_done_flags', {})
for exp_name in experiments.keys():
    done, completed, target = get_train_status(exp_name)
    train_done_flags[exp_name] = done
    state = 'DONE' if done else 'PENDING'
    print(f'[{state}] {exp_name}: {completed}/{target} epochs')

In [ ]:
exp_name = 'yolo11n'
if train_done_flags.get(exp_name, False):
    done, completed, target = get_train_status(exp_name)
    print(f'[SKIP] {exp_name} đã train xong ({completed}/{target} epochs).')
else:
    ckpt = experiments[exp_name]
    result = train_or_resume(exp_name, ckpt)
    train_runs[exp_name] = result
    done, completed, target = get_train_status(exp_name)
    train_done_flags[exp_name] = done
    print(f'[STATUS] {exp_name}: {completed}/{target} epochs | done={done}')

print(train_done_flags)
train_runs

In [ ]:
exp_name = 'yolo11s'
if train_done_flags.get(exp_name, False):
    done, completed, target = get_train_status(exp_name)
    print(f'[SKIP] {exp_name} đã train xong ({completed}/{target} epochs).')
else:
    ckpt = experiments[exp_name]
    result = train_or_resume(exp_name, ckpt)
    train_runs[exp_name] = result
    done, completed, target = get_train_status(exp_name)
    train_done_flags[exp_name] = done
    print(f'[STATUS] {exp_name}: {completed}/{target} epochs | done={done}')

print(train_done_flags)
train_runs

## **6. Đánh giá mô hình trên tập validation**
Báo cáo các metric chuẩn cho bài toán object detection: Precision, Recall, mAP@0.5, mAP@0.5:0.95 và tốc độ suy luận.

In [ ]:
def get_run_info(exp_name):
    run_name = f'{exp_name}_coco'
    run_dir = PROJECT_DIR / run_name
    best_weight = run_dir / 'weights' / 'best.pt'
    last_weight = run_dir / 'weights' / 'last.pt'
    return {
        'model': exp_name,
        'run_dir': run_dir,
        'best_weight': best_weight,
        'last_weight': last_weight,
        'best_exists': best_weight.exists(),
        'last_exists': last_weight.exists(),
    }


def evaluate_run(exp_name, run_info):
    if not run_info['best_exists']:
        print(f"[SKIP] {exp_name}: chưa có best.pt tại {run_info['best_weight']}")
        return None

    model = YOLO(str(run_info['best_weight']))

    run_detections_dir = DETECTIONS_DIR / f'{exp_name}_coco'
    run_detections_dir.mkdir(parents=True, exist_ok=True)

    metrics = model.val(
        data=DATA_YAML,
        split='val',
        save_json=True,
        plots=True,
        project=str(run_detections_dir.parent),
        name=run_detections_dir.name,
    )
    return {
        'model': exp_name,
        'run_dir': str(run_info['run_dir']),
        'best_weight': str(run_info['best_weight']),
        'precision': float(metrics.box.mp),
        'recall': float(metrics.box.mr),
        'mAP50': float(metrics.box.map50),
        'mAP50-95': float(metrics.box.map),
        'inference_ms_per_image': float(metrics.speed.get('inference', float('nan'))),
        'val_dir': str(metrics.save_dir)
    }


run_registry = {name: get_run_info(name) for name in experiments.keys()}
records = []

run_root = PROJECT_DIR
REPORT_DIR = run_root / 'report_artifacts'
FIG_DIR = REPORT_DIR / 'figures'
TABLE_DIR = REPORT_DIR / 'tables'
DETECTIONS_DIR = REPORT_DIR / 'detections'
for d in [REPORT_DIR, FIG_DIR, TABLE_DIR, DETECTIONS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

for name, info in run_registry.items():
    rec = evaluate_run(name, info)
    if rec is not None:
        records.append(rec)

result_df = pd.DataFrame(records)
if not result_df.empty:
    result_df = result_df.sort_values('mAP50-95', ascending=False).reset_index(drop=True)

metrics_json = TABLE_DIR / 'evaluation_records.json'
with open(metrics_json, 'w', encoding='utf-8') as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

display(result_df)
print(f'Đã tạo thư mục báo cáo: {REPORT_DIR.resolve()}')
print(f'Đã lưu metrics raw: {metrics_json.resolve()}')

In [ ]:
if result_df.empty:
    print('Chưa có run nào có best.pt để vẽ biểu đồ. Hãy train xong ít nhất 1 mô hình rồi chạy lại cell này.')
else:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    sns.barplot(data=result_df, x='model', y='mAP50', ax=axes[0], palette='Set2')
    axes[0].set_title('mAP@0.5')

    sns.barplot(data=result_df, x='model', y='mAP50-95', ax=axes[1], palette='Set2')
    axes[1].set_title('mAP@0.5:0.95')

    sns.barplot(data=result_df, x='model', y='inference_ms_per_image', ax=axes[2], palette='Set2')
    axes[2].set_title('Inference time (ms/image)')

    for ax in axes:
        ax.set_xlabel('Model')
        ax.tick_params(axis='x', rotation=0)

    plt.tight_layout()
    metrics_plot_path = FIG_DIR / 'metrics_comparison.png'
    fig.savefig(metrics_plot_path, dpi=200, bbox_inches='tight')
    plt.show()
    print(f'Đã lưu biểu đồ so sánh: {metrics_plot_path.resolve()}')

Nhìn vào bảng và biểu đồ, có thể đánh giá rõ trade-off giữa độ chính xác và tốc độ giữa hai biến thể YOLOv11.

Mô hình có mAP cao hơn chưa chắc là lựa chọn tốt nhất trong mọi bối cảnh; với các bài toán thời gian thực, độ trễ suy luận là yếu tố cần cân nhắc tương đương.

## **7. Trực quan kết quả dự đoán**
Minh họa detection trên một số ảnh validation để quan sát trực tiếp chất lượng localization và classification.

In [ ]:
if 'DATA_YAML' not in globals():
    USE_COCO128_DEBUG = globals().get('USE_COCO128_DEBUG', False)
    DATA_YAML = 'coco128.yaml' if USE_COCO128_DEBUG else 'coco.yaml'

if 'PROJECT_DIR' not in globals():
    local_runs = globals().get('RUNS_ROOT', resolve_runs_root())
    PROJECT_DIR = Path(local_runs) / 'assignment2_yolov11_coco'

if 'FIG_DIR' not in globals():
    FIG_DIR = PROJECT_DIR / 'report_artifacts' / 'figures'
    FIG_DIR.mkdir(parents=True, exist_ok=True)

if 'result_df' not in globals():
    fallback_experiments = globals().get('experiments', {'yolo11n': 'yolo11n.pt', 'yolo11s': 'yolo11s.pt'})
    rows = []
    for exp_name in fallback_experiments.keys():
        best_weight = PROJECT_DIR / f'{exp_name}_coco' / 'weights' / 'best.pt'
        if best_weight.exists():
            rows.append({'model': exp_name, 'best_weight': str(best_weight)})
    result_df = pd.DataFrame(rows)

if 'val_images' not in globals():
    local_data_info = data_info if 'data_info' in globals() else check_det_dataset(DATA_YAML)
    if 'resolve_path' in globals():
        val_images_path = resolve_path(local_data_info.get('path', '.'), local_data_info['val'])
    else:
        p = Path(local_data_info['val'])
        val_images_path = p if p.is_absolute() else (Path(local_data_info.get('path', '.')) / p)

    if val_images_path.is_file() and val_images_path.suffix.lower() == '.txt':
        val_images = []
        with open(val_images_path, 'r', encoding='utf-8') as f:
            for line in f:
                s = line.strip()
                if not s:
                    continue
                p = Path(s)
                if not p.is_absolute():
                    p = Path(local_data_info.get('path', '.')) / p
                if p.exists():
                    val_images.append(p)
        val_images = sorted(val_images)
    else:
        val_images = sorted(
            list(val_images_path.glob('*.jpg')) +
            list(val_images_path.glob('*.jpeg')) +
            list(val_images_path.glob('*.png'))
        )

if result_df.empty:
    print('Không có kết quả đánh giá để trực quan. Hãy chạy Section 6 sau khi đã có best.pt.')
elif 'best_weight' not in result_df.columns:
    print('Thiếu cột best_weight trong result_df. Hãy chạy lại Section 6 trước.')
elif len(val_images) == 0:
    print('Không tìm thấy ảnh validation để trực quan hóa.')
else:
    best_model_name = str(result_df.iloc[0]['model'])
    best_model_path = Path(result_df.iloc[0]['best_weight'])
    if not best_model_path.exists():
        print(f'Không tìm thấy best.pt tại {best_model_path}. Hãy train/evaluate lại Section 5-6.')
    else:
        best_model = YOLO(str(best_model_path))
        sample_images = random.sample(val_images, k=min(8, len(val_images)))
        pred_results = best_model.predict(source=[str(p) for p in sample_images], conf=0.25, save=False, verbose=False)

        pred_dir = FIG_DIR / 'predictions'
        pred_dir.mkdir(parents=True, exist_ok=True)

        fig, axes = plt.subplots(2, 4, figsize=(18, 9))
        axes = axes.flatten()
        for i, (ax, pred) in enumerate(zip(axes, pred_results), start=1):
            plotted = pred.plot()[:, :, ::-1]
            ax.imshow(plotted)
            ax.axis('off')
            plt.imsave(pred_dir / f'{best_model_name}_sample_{i:02d}.png', plotted)

        for i in range(len(pred_results), len(axes)):
            axes[i].axis('off')

        plt.suptitle(f'Visualization - {best_model_name}', fontsize=14)
        plt.tight_layout()
        grid_path = FIG_DIR / f'visualization_grid_{best_model_name}.png'
        fig.savefig(grid_path, dpi=200, bbox_inches='tight')
        plt.show()

        print(f'Đã lưu grid visualize: {grid_path.resolve()}')
        print(f'Đã lưu ảnh prediction riêng lẻ tại: {pred_dir.resolve()}')

Quan sát định tính giúp bổ sung cho các metric định lượng, đặc biệt trong các trường hợp mô hình phát hiện đúng nhưng confidence thấp hoặc nhầm giữa các lớp gần nhau.

Kết hợp hai góc nhìn định lượng và định tính sẽ giúp phần thảo luận kết quả trong báo cáo thuyết phục hơn.

## **8. Tổng hợp kết quả cho báo cáo**

In [ ]:
if 'PROJECT_DIR' not in globals():
    local_runs = globals().get('RUNS_ROOT', resolve_runs_root())
    PROJECT_DIR = Path(local_runs) / 'assignment2_yolov11_coco'

REPORT_DIR = globals().get('REPORT_DIR', PROJECT_DIR / 'report_artifacts')
TABLE_DIR = globals().get('TABLE_DIR', REPORT_DIR / 'tables')
REPORT_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

if 'result_df' not in globals() or result_df is None:
    result_df = pd.DataFrame()

experiments = globals().get('experiments', {'yolo11n': 'yolo11n.pt', 'yolo11s': 'yolo11s.pt'})
TRAIN_ARGS = globals().get('TRAIN_ARGS', {})

if 'run_registry' not in globals() or run_registry is None:
    run_registry = {}
    for exp_name in experiments.keys():
        run_dir = PROJECT_DIR / f'{exp_name}_coco'
        best_weight = run_dir / 'weights' / 'best.pt'
        last_weight = run_dir / 'weights' / 'last.pt'
        run_registry[exp_name] = {
            'run_dir': run_dir,
            'best_exists': best_weight.exists(),
            'last_exists': last_weight.exists(),
        }

summary_cols = ['model', 'precision', 'recall', 'mAP50', 'mAP50-95', 'inference_ms_per_image']
if result_df.empty:
    summary_df = pd.DataFrame(columns=summary_cols)
else:
    summary_df = result_df.reindex(columns=summary_cols).copy()
    summary_df = summary_df.round({
        'precision': 4,
        'recall': 4,
        'mAP50': 4,
        'mAP50-95': 4,
        'inference_ms_per_image': 2
    })

summary_csv = TABLE_DIR / 'comparison_metrics.csv'
summary_json = TABLE_DIR / 'comparison_metrics.json'
run_meta_yaml = TABLE_DIR / 'run_metadata.yaml'

summary_df.to_csv(summary_csv, index=False)
summary_df.to_json(summary_json, orient='records', force_ascii=False, indent=2)

run_metadata = {
    'dataset_yaml': globals().get('DATA_YAML', 'coco.yaml'),
    'experiments': experiments,
    'train_args': TRAIN_ARGS,
    'run_dirs': {k: str(v['run_dir']) for k, v in run_registry.items()},
    'best_weight_exists': {k: bool(v.get('best_exists', False)) for k, v in run_registry.items()},
    'last_weight_exists': {k: bool(v.get('last_exists', False)) for k, v in run_registry.items()}
}
with open(run_meta_yaml, 'w', encoding='utf-8') as f:
    yaml.safe_dump(run_metadata, f, sort_keys=False, allow_unicode=True)

display(summary_df)
print(f'Đã lưu bảng CSV: {summary_csv.resolve()}')
print(f'Đã lưu bảng JSON: {summary_json.resolve()}')
print(f'Đã lưu metadata run: {run_meta_yaml.resolve()}')

## **9. Hướng mở rộng**
- So sánh thêm với mô hình khác họ one-stage/two-stage (VD: YOLO11 vs Faster R-CNN/RT-DETR).
- Phân tích lỗi FN/FP theo từng lớp và confusion matrix.
- Đánh giá trade-off tốc độ/chính xác trên nhiều kích thước model (n/s/m/l).
- Demo real-time với webcam/video và triển khai giao diện Gradio/Streamlit.

